# EVE 310 - Lab 11: Comparing regression models

**Module 2 | 11/12/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab11-dairy-farm-challenge/notebooks/lab11-tutorial.ipynb)

## Learning objectives

1. Compute the least-squares betas with the normal equation in NumPy. Check that `LinearRegression` agrees.
2. Score a model on the testing days with MSE, MAE and R². Say what each measures, in which units.
3. Fit a random forest. Spot overfitting from its training and testing errors.
4. Build a new column from existing ones. Judge if a simple model with the right column beats a more complex one.

## Before you start

1. Click **Copy to Drive** (top of this window) and work in the copy.
2. Run the setup cell below first.
3. **Type along** with the instructor. Five cells have `--` blanks. In each, type what the **Type it** line above says, then run the cell. Stuck? Open **Show the finished cell** below it. **Run all** fails until every `--` is filled. Other cells say **Run as is**.
4. **Predict first** (three cells): decide what the cell will print before you run it. In lab, answer on Menti.
5. Nothing to submit here. Optional: later, download the **activity** notebook as `.ipynb` and upload it to Gradescope for feedback.

<details><summary>Why?</summary>

If you do not work in a copy, Colab loses your changes when the session ends. The setup cell downloads the two data files into `DATA_DIR` and creates `FIGURES_DIR`. Each typed cell has one or two blanks. The finished cell helps if you get stuck or work on your own. A cell that still has a `--` in it cannot run. A wrong guess on a **Predict first** line is useful. It shows you what to look for in the output.

</details>

### Tutorial

A city utility must forecast tomorrow's peak electricity demand to decide which power plants to run. `power_demand.csv` has one year for a Texas city, made up for this lab. Each row is one day: its average temperature and its peak demand in megawatts (MW). Demand rises on hot days for air conditioning and on cold days for electric heating.

**Which model predicts best on days it has not seen, and can you say why?** Three models are fitted on the same 255 **training days**. Each is scored on the same 110 **testing days**, which it does not see during fitting:

- **straight line**: the normal equation in NumPy, then `LinearRegression`
- **random forest**: no equation and no betas
- **curve**: the line plus one column, the temperature squared

**The plan.** Every lab follows the same six steps, named in the section titles.

| Step | What it means | In this tutorial | In the activity |
| --- | --- | --- | --- |
| **Load** | read the file into a dataframe | section 1 | the data cell |
| **Inspect** | look at the rows and a plot before any model | section 1 | the data cell |
| **Prepare** | choose `x` and `y`, split, build any new column | sections 1 and 5 | the data cell, Question 3 |
| **Fit** | create the model, then `fit` it on the training rows | sections 2, 4 and 5 | Questions 1 and 3 |
| **Check** | score the model on the testing rows and compare | sections 3 to 5 | Questions 2 and 3 |
| **Interpret** | read the results in the units of the problem | section 6 | the written answers |

Stuck on a blank cell? Find its step in the table. Ask: what goes into this step, and what comes out?

## 0. Setup

In [ ]:
# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab11-dairy-farm-challenge"
DATA_FILES = ["power_demand.csv", "Dairy-Farm-Dataset.csv"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

**Run as is:** the imports. Three are new: `RandomForestRegressor`, `mean_squared_error` and `mean_absolute_error`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## 1. Load, inspect, prepare: a year of temperature and demand

**Run as is.** The cell **loads** the file, **inspects** it with `describe()` and a scatter plot, and **prepares** `x`, `y` and one 70/30 split. Every model uses this split, so we can compare their scores.

<details><summary>Why?</summary>

`describe()` shows the range of each column: its smallest and largest values. The scatter plot shows the shape that every model must follow. `x` is `demand_df[['temp_f']]`. The two pairs of brackets give a dataframe with one column. It has one row per day and one column per input, the shape scikit-learn wants. `train_test_split` uses 70% of the days for fitting. It keeps 30% aside for testing, as in Lab 5.

</details>

In [ ]:
# Load: one row per day
demand_df = pd.read_csv(DATA_DIR / 'power_demand.csv')
print(len(demand_df), 'days')
print(demand_df.describe().round(1))

# Inspect: demand against temperature
fig, ax = plt.subplots()
ax.scatter(demand_df['temp_f'], demand_df['demand_mw'], s=12, color='k')
ax.set_xlabel('average temperature (°F)')
ax.set_ylabel('peak demand (MW)')
ax.set_title('365 days')

# Prepare: the input, the column to predict, and the 70/30 split
x = demand_df[['temp_f']]
y = demand_df['demand_mw']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=0)
print(len(x_train), 'training days,', len(x_test), 'testing days')

Demand is lowest, near 1,400 MW, on mild days in the upper 50s °F. It rises on colder and on hotter days, much faster on hotter days. No straight line can follow this curve.

## 2. Fit: the normal equation, then `LinearRegression`

The straight line is demand = β0 + β1 × temperature. The lecture's **normal equation** is β = (XᵀX)⁻¹Xᵀy. It gives the betas with the smallest sum of squared errors (SSE) on the training days. Each symbol is one variable in the cell:

| Lecture | Code | Shape here |
| --- | --- | --- |
| X, the design matrix | `X1` | 255 × 2 |
| XᵀX | `xtx` | 2 × 2 |
| (XᵀX)⁻¹ | `inv_xtx` | 2 × 2 |
| Xᵀy | `xty` | 2 |
| β | `beta` | 2 |

`X1` is the **design matrix**: one row per training day, one column per beta. It starts with a **column of ones**. On every row, β0 multiplies the 1 in this column, so β0 is the intercept. Next come the temperatures. `np.ones(len(x_train))` makes one 1 per training day. `np.c_[a, b]` puts columns side by side.

**Type it:** `X1 = np.c_[np.ones(len(x_train)), x_train]`, then `beta = np.dot(inv_xtx, xty)`.

In [ ]:
# The design matrix: a column of ones for the intercept, then the temperatures
--
print(X1.shape, '= (training days, columns)')

# The normal equation, beta = (X^T X)^(-1) X^T y, in the lecture notebook's four steps
xtx = np.dot(X1.T, X1)          # X-transpose times X
inv_xtx = np.linalg.inv(xtx)    # its inverse
xty = np.dot(X1.T, y_train)     # X-transpose times y
--
print('xtx =', np.round(xtx).astype(int).tolist())
print('xty =', np.round(xty).astype(int).tolist())
print('beta from the normal equation:', np.round(beta, 4))

<details><summary>Show the finished cell</summary>

```python
# The design matrix: a column of ones for the intercept, then the temperatures
X1 = np.c_[np.ones(len(x_train)), x_train]
print(X1.shape, '= (training days, columns)')

# The normal equation, beta = (X^T X)^(-1) X^T y, in the lecture notebook's four steps
xtx = np.dot(X1.T, X1)          # X-transpose times X
inv_xtx = np.linalg.inv(xtx)    # its inverse
xty = np.dot(X1.T, y_train)     # X-transpose times y
beta = np.dot(inv_xtx, xty)
print('xtx =', np.round(xtx).astype(int).tolist())
print('xty =', np.round(xty).astype(int).tolist())
print('beta from the normal equation:', np.round(beta, 4))
```

</details>

`xtx` holds four sums over the 255 training days. The top-left entry is the number of days. The sum of the temperatures appears twice. The last entry is the sum of the squared temperatures. The normal equation uses these sums to make two equations with two unknowns. `beta` is the solution: β0, the intercept in MW, then β1, the slope in MW per °F.

<details><summary>More: xty</summary>

`xty` holds the sum of the demands and the sum of temperature × demand. The betas come out in the same order as the columns of `X1`.

</details>

Next, fit the same line with `LinearRegression`, as in Lab 5. Then compute the training SSE of two tilted lines: one 1 MW per °F steeper than `beta`, one 1 MW per °F flatter.

**Predict first:** compared with the SSE of `beta`, will the steeper line's SSE be larger, smaller, or the same? And the flatter line's?

**Type it:** the SSE of `beta`, `sse_best = np.sum((y_train - np.dot(X1, beta)) ** 2)`.

In [ ]:
lm = LinearRegression()
lm.fit(x_train, y_train)
print('intercept_ and coef_:', round(lm.intercept_, 4), np.round(lm.coef_, 4))

# The SSE on the training days: subtract the line's predictions, square, add up
--
# The same sum for two lines tilted 1 MW per °F away from beta
sse_steeper = np.sum((y_train - np.dot(X1, beta + np.array([0, 1]))) ** 2)
sse_flatter = np.sum((y_train - np.dot(X1, beta - np.array([0, 1]))) ** 2)
print('SSE of beta:               ', round(sse_best))
print('SSE, slope 1 MW/°F steeper:', round(sse_steeper))
print('SSE, slope 1 MW/°F flatter:', round(sse_flatter))
# X1 transposed times the residuals of beta: the normal equation says this is zero
print('X1.T times the residuals:', np.round(np.dot(X1.T, y_train - np.dot(X1, beta)), 3) + 0)

<details><summary>Show the finished cell</summary>

```python
lm = LinearRegression()
lm.fit(x_train, y_train)
print('intercept_ and coef_:', round(lm.intercept_, 4), np.round(lm.coef_, 4))

# The SSE on the training days: subtract the line's predictions, square, add up
sse_best = np.sum((y_train - np.dot(X1, beta)) ** 2)
# The same sum for two lines tilted 1 MW per °F away from beta
sse_steeper = np.sum((y_train - np.dot(X1, beta + np.array([0, 1]))) ** 2)
sse_flatter = np.sum((y_train - np.dot(X1, beta - np.array([0, 1]))) ** 2)
print('SSE of beta:               ', round(sse_best))
print('SSE, slope 1 MW/°F steeper:', round(sse_steeper))
print('SSE, slope 1 MW/°F flatter:', round(sse_flatter))
# X1 transposed times the residuals of beta: the normal equation says this is zero
print('X1.T times the residuals:', np.round(np.dot(X1.T, y_train - np.dot(X1, beta)), 3) + 0)
```

</details>

`intercept_` and `coef_` match `beta`. So `fit` **is** the normal equation. Both tilted lines have a larger SSE. Each is 1,256,778 MW² larger. Every other line has a larger SSE than `beta`. That is what "least squares" means.

A **residual** is a training day's error: its demand minus the line's prediction. The last printed line shows Xᵀ(y − Xβ) = 0, as the normal equation says. For the column of ones, this means the residuals add up to zero.

<details><summary>Why?</summary>

You can rewrite XᵀXβ = Xᵀy as Xᵀ(y − Xβ) = 0. It means: multiply the residuals by each column of `X1`, add up, and you get zero. For the temperature column, it means no straight-line trend with temperature is left in the residuals. A curved trend can be left, as section 3 shows. If you tilt the line, this balance breaks and the SSE goes up. scikit-learn solves the same equation without computing the inverse. Its method is less affected by rounding. The printed digits are the same.

</details>

## 3. Check: MSE, MAE and R² on the testing days

Each testing day has one **error**, `y_test - y_pred`, in MW. Three scores describe these errors:

- **MSE**, mean squared error: square each error, then average. In MW². Squaring makes large misses count much more than small ones.
- **MAE**, mean absolute error: drop the sign of each error with `np.abs`, then average. In MW: "off by this much on a typical day".
- **R²**, from Lab 5: the share of the day-to-day variation in `y` that the model explains. No units.

A smaller MSE or MAE is better. A larger R² is better.

<details><summary>More: the MSE</summary>

The MSE is the SSE divided by the number of days. The normal equation made this number as small as possible on the training days. Its unit, MW², is hard to read. The MAE is in MW, so it is easy to read.

</details>

**Type it:** `mae = np.mean(np.abs(y_test - y_pred))`.

In [ ]:
# One prediction per testing day
y_pred = lm.predict(x_test)

# Mean squared error: the SSE of the testing days divided by the number of days
mse = np.mean((y_test - y_pred) ** 2)
# Mean absolute error: the errors with their signs dropped, averaged
--
r2 = r2_score(y_test, y_pred)
print('MSE:', round(mse, 1), 'MW squared')
print('MAE:', round(mae, 1), 'MW')
print('R2: ', round(r2, 3))
# scikit-learn's functions give the same two numbers
print('scikit-learn:', round(mean_squared_error(y_test, y_pred), 1),
      round(mean_absolute_error(y_test, y_pred), 1))

<details><summary>Show the finished cell</summary>

```python
# One prediction per testing day
y_pred = lm.predict(x_test)

# Mean squared error: the SSE of the testing days divided by the number of days
mse = np.mean((y_test - y_pred) ** 2)
# Mean absolute error: the errors with their signs dropped, averaged
mae = np.mean(np.abs(y_test - y_pred))
r2 = r2_score(y_test, y_pred)
print('MSE:', round(mse, 1), 'MW squared')
print('MAE:', round(mae, 1), 'MW')
print('R2: ', round(r2, 3))
# scikit-learn's functions give the same two numbers
print('scikit-learn:', round(mean_squared_error(y_test, y_pred), 1),
      round(mean_absolute_error(y_test, y_pred), 1))
```

</details>

**Run as is:** draw the line over the testing days.

<details><summary>More: grid</summary>

`grid` is a dataframe of temperatures from 44 to 94 °F. It has the same column name as `x`, so every model can draw its predictions as a line.

</details>

In [ ]:
# Temperatures from 44 to 94 °F, half a degree apart, to draw each model as a line
grid = pd.DataFrame({'temp_f': np.arange(44, 94.5, 0.5)})
fig, ax = plt.subplots()
ax.scatter(x_test['temp_f'], y_test, s=12, color='k', label='testing days')
ax.plot(grid['temp_f'], lm.predict(grid), color='tab:blue', label='straight line')
ax.set_xlabel('average temperature (°F)')
ax.set_ylabel('peak demand (MW)')
ax.legend()

The line is off by 143 MW on a typical testing day. Its errors follow a pattern: at both ends, the points curve away from the line. An MSE of about 29,850 MW² tells you little on its own. Use it to compare models on the same testing days.

<details><summary>More: the errors</summary>

The line predicts too little on cold days and on the hottest days. It predicts too much in the 60s and 70s °F. Sections 4 and 5 compare it with two other models on the same testing days.

</details>

## 4. Fit and check a different model: a random forest

A **decision tree** asks yes/no questions learned from the training days, for example: *is the temperature above 78.85 °F? If yes, is it above 86.6 °F?* Each path of answers ends in a **leaf**. The tree predicts the average demand of the training days in that leaf. A **random forest** grows 100 such trees and averages their predictions. Each tree grows from a **resample**: 255 training days drawn at random, some more than once. `random_state=0` makes the resamples the same every time, so everyone gets the same forest.

<details><summary>More: the forest</summary>

Each tree asks many more than two questions. The steps are the ones you know: create the model, `fit` it on the training days, then `predict`. In Lab 10, `random_state` kept the random starts the same in this way.

</details>

**Predict first:** a forest can grow a leaf for almost every training day. Will its MAE on its own training days be about the same as on the testing days, or much smaller? And the line's?

**Type it:** `rf = RandomForestRegressor(random_state=0)`, then `rf.fit(x_train, y_train)`.

In [ ]:
# A random forest: 100 trees, each grown on a random resample of the training days
--
--

# Each model's MAE on the days it was fitted on, and on the testing days
y_fit_rf = rf.predict(x_train)
y_pred_rf = rf.predict(x_test)
print('line:   MAE training days', round(mean_absolute_error(y_train, lm.predict(x_train)), 1),
      'MW, testing days', round(mae, 1), 'MW')
print('forest: MAE training days', round(mean_absolute_error(y_train, y_fit_rf), 1),
      'MW, testing days', round(mean_absolute_error(y_test, y_pred_rf), 1), 'MW')

<details><summary>Show the finished cell</summary>

```python
# A random forest: 100 trees, each grown on a random resample of the training days
rf = RandomForestRegressor(random_state=0)
rf.fit(x_train, y_train)

# Each model's MAE on the days it was fitted on, and on the testing days
y_fit_rf = rf.predict(x_train)
y_pred_rf = rf.predict(x_test)
print('line:   MAE training days', round(mean_absolute_error(y_train, lm.predict(x_train)), 1),
      'MW, testing days', round(mae, 1), 'MW')
print('forest: MAE training days', round(mean_absolute_error(y_train, y_fit_rf), 1),
      'MW, testing days', round(mean_absolute_error(y_test, y_pred_rf), 1), 'MW')
```

</details>

**Run as is:** draw the forest and the line over the testing days.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x_test['temp_f'], y_test, s=12, color='k', label='testing days')
ax.plot(grid['temp_f'], lm.predict(grid), color='tab:blue', label='straight line')
ax.plot(grid['temp_f'], rf.predict(grid), color='tab:orange', label='random forest')
ax.set_xlabel('average temperature (°F)')
ax.set_ylabel('peak demand (MW)')
ax.legend()

The forest follows the curve in small steps. Its testing MAE is half the line's: 72 MW against 143 MW. Now compare each model with itself, its training MAE against its testing MAE:

- line: training 141 MW, testing 143 MW, about the same.
- forest: training 35 MW, testing 72 MW. It has partly memorized its training days, with their **noise**: random ups and downs that new days do not repeat.

That is **overfitting**: a model much better on its training days than on new days. So we judge models on the testing days only.

Compare training and testing MAEs, not R². R² also depends on how spread out the demands are in each set of days. The testing days' demands are less spread out, and that lowers every model's testing R².

<details><summary>More: the gap</summary>

The forest still wins on the testing days here. The gap between its training MAE and its testing MAE tells you how much to trust its training score.

</details>

## 5. Your turn (prepare, fit, check): one more column, and the line becomes a curve

The new column is `temp_sq`, the temperature squared. The model is still `LinearRegression` and the normal equation, now with three betas: demand = β0 + β1 × temperature + β2 × temperature². Each beta multiplies one column, so it is still linear regression, with a curved prediction.

`x_train_sq` and `x_test_sq` are **copies** of `x_train` and `x_test`, made with `.copy()`. Adding a column to a copy leaves `x_train` and `x_test` unchanged for the line and the forest. Fill each `--` with the temperatures of the same days, squared with `** 2` as in section 2. The testing copy needs the column too, because a model fitted on two columns needs the same two to predict.

<details><summary>More: new columns</summary>

Lab 6 built new columns from old ones. There, `Temp_x_Med` was the product of two columns.

</details>

In [ ]:
# Copies of x_train and x_test with a second column, the temperature squared
x_train_sq = x_train.copy()
x_train_sq['temp_sq'] = --
x_test_sq = x_test.copy()
x_test_sq['temp_sq'] = --

lm_sq = LinearRegression()
lm_sq.fit(x_train_sq, y_train)
y_pred_sq = lm_sq.predict(x_test_sq)
print('betas:', round(lm_sq.intercept_, 2), np.round(lm_sq.coef_, 5))
print('curve:  MAE training days', round(mean_absolute_error(y_train, lm_sq.predict(x_train_sq)), 1),
      'MW, testing days', round(mean_absolute_error(y_test, y_pred_sq), 1), 'MW')

<details><summary>Show the finished cell</summary>

```python
# Copies of x_train and x_test with a second column, the temperature squared
x_train_sq = x_train.copy()
x_train_sq['temp_sq'] = x_train['temp_f'] ** 2
x_test_sq = x_test.copy()
x_test_sq['temp_sq'] = x_test['temp_f'] ** 2

lm_sq = LinearRegression()
lm_sq.fit(x_train_sq, y_train)
y_pred_sq = lm_sq.predict(x_test_sq)
print('betas:', round(lm_sq.intercept_, 2), np.round(lm_sq.coef_, 5))
print('curve:  MAE training days', round(mean_absolute_error(y_train, lm_sq.predict(x_train_sq)), 1),
      'MW, testing days', round(mean_absolute_error(y_test, y_pred_sq), 1), 'MW')
```

</details>

**Run as is:** draw the three models over the testing days. The curve, `lm_sq`, needs `grid` with the squared column too.

In [ ]:
grid_sq = grid.copy()
grid_sq['temp_sq'] = grid['temp_f'] ** 2
fig, ax = plt.subplots()
ax.scatter(x_test['temp_f'], y_test, s=12, color='k', label='testing days')
ax.plot(grid['temp_f'], lm.predict(grid), color='tab:blue', label='straight line')
ax.plot(grid['temp_f'], rf.predict(grid), color='tab:orange', label='random forest')
ax.plot(grid['temp_f'], lm_sq.predict(grid_sq), color='tab:green', label='curve')
ax.set_xlabel('average temperature (°F)')
ax.set_ylabel('peak demand (MW)')
ax.legend()

The curve's testing MAE is 58 MW. That beats the forest's 72 MW. The curve's equation also fits on one line. It does not overfit: 60 MW on training days, 58 MW on testing days. The forest had to find the curve step by step. The squared column gives the shape directly to a simpler model.

## 6. Interpret: the scores side by side, and two new days

**Run as is.** The cell prints the scores side by side. Then it predicts the demand on a mild 75 °F day and on a 105 °F heat wave. The heat wave is hotter than any day in the file.

<details><summary>More: the new days</summary>

`new_days` is a dataframe with the same column name as `x`. `new_days_sq` adds the squared column, because `lm_sq` was fitted on both columns.

</details>

**Predict first:** for the 105 °F day, which model will predict the highest demand, and which the lowest?

In [ ]:
scores_df = pd.DataFrame({
    'model': ['straight line', 'random forest', 'curve'],
    'MAE training (MW)': [mean_absolute_error(y_train, lm.predict(x_train)), mean_absolute_error(y_train, y_fit_rf),
                          mean_absolute_error(y_train, lm_sq.predict(x_train_sq))],
    'MAE testing (MW)': [mae, mean_absolute_error(y_test, y_pred_rf), mean_absolute_error(y_test, y_pred_sq)],
    'R2 testing': [r2, r2_score(y_test, y_pred_rf), r2_score(y_test, y_pred_sq)],
})
print(scores_df.round(3))

# Two new days: a mild one, and a heat wave hotter than any day in the data
new_days = pd.DataFrame({'temp_f': [75.0, 105.0]})
new_days_sq = new_days.copy()
new_days_sq['temp_sq'] = new_days['temp_f'] ** 2
print('straight line:', np.round(lm.predict(new_days)))
print('random forest:', np.round(rf.predict(new_days)))
print('curve:        ', np.round(lm_sq.predict(new_days_sq)))

At 75 °F the three models agree within about 150 MW. At 105 °F they are far apart: line about 2,770 MW, curve about 3,950 MW, forest 2,876 MW.

- **Forest**: its predictions are averages of training-day demands. So it never predicts above the highest training demand, 2,916 MW. Every day hotter than the hottest training day, 93.6 °F, gets the same 2,876 MW.
- **Line and curve**: they keep their shapes past the data. No model has seen a 105 °F day, so their forecasts there are guesses.

Every model is least trustworthy outside its training range: colder than its coldest training day, or hotter than its hottest. On those days, a utility needs the forecast most.

**Three things to remember**

1. **`fit` is the normal equation.** A squared column, or a product of two columns, is one more column of `X`, so it is still linear regression.
2. **Judge a model on the testing days, in units someone can act on.** MAE is in the units of `y`. MSE is in squared units and makes big misses count more. R² has no units. A training error much smaller than the testing error means overfitting.
3. **The right column can beat a more complex model**, and it keeps betas you can explain. No model is safe outside its training range. A forest never predicts above the largest or below the smallest training `y`.

<details><summary><b>Common mistakes</b>: open this when a cell fails</summary>

| What you see | What went wrong |
| --- | --- |
| `TypeError: 'CClass' object is not callable` | You wrote `np.c_(...)` with round brackets. `np.c_` takes square brackets: `np.c_[a, b]` |
| `TypeError: Cannot interpret ... as a data type` | A bracket is in the wrong place: `np.ones(len(x_train), x_train)`. Close `np.ones(len(x_train))` before the comma |
| `ValueError: all the input array dimensions except for the concatenation axis must match exactly` | The ones and `x_train` have different lengths. For example, `np.ones(len(x))` gives 365 ones for 255 training days |
| `beta` prints huge numbers, such as `[5.72316292e+11 4.08692237e+13]` | You wrote `np.dot(xtx, xty)`, so the inverse is missing. The first factor must be `inv_xtx`, (XᵀX)⁻¹ |
| `beta` has two numbers, `lm.coef_` only one | Not a mistake. `beta[0]` is `lm.intercept_` and `beta[1]` is `lm.coef_[0]`. `coef_` holds only the slopes |
| `ValueError: operands could not be broadcast together with shapes (255,) (110,)` | One line mixes training and testing arrays, such as `y_train - y_pred`: 255 training demands and 110 predictions. The testing errors are `y_test - y_pred` |
| The MAE is near 0, or negative | `np.abs` is missing. In `np.mean(y_test - y_pred)`, the positive and negative errors cancel |
| The forest's numbers differ from your neighbor's | `random_state=0` is missing, so your forest grew from other random resamples |
| `TypeError: Cannot perform 'xor' with a dtyped [float64] array and scalar of type [bool]` | `^ 2` is not a power in Python. Square with `** 2` |
| `ValueError: Input X contains NaN` | A copy's column was built from the other set of days, such as `x_test_sq['temp_sq'] = x_train['temp_f'] ** 2`. pandas matches rows by their index. No testing day is a training day, so every value is missing (NaN). Build each copy's column from its own days |

</details>

## 7. Why this matters

- **A forecast is a decision.** It tells the utility which power plants to run. The most costly misses come on the hottest and coldest days. There the straight line is worst. Past its training range, the forest's prediction stays flat.
- **Betas explain, forests predict.** Betas say *how much* the prediction changes with each column. In the activity, they say how much each farm practice changes the predicted carbon footprint of milk. A forest only says *what* the prediction is. Sometimes that is all you need, and sometimes not.

Now open the **activity** notebook.

## Wrap-up

Keep this notebook open. In the activity, you fit a linear model and a forest on 7,000 dairy farms and score them on the other 3,000. You also build a column from the units of the data, not from a plot. Question 3 gives only the steps as comments, and you write the code. Use the step table above.

At home: in the [least-squares explorer](https://thyanrevolter.github.io/eve310-fall-2026/labs/lab11-least-squares-explorer.html), drag a line through this tutorial's training days. Watch how the residuals and the SSE change.